# Kubernetes Basics — Pods, Logs & Debugging

**[website version](https://training.nrp-nautilus.io/cms-hats/2_kubernetes_basics.html)** — run cells with **Shift+Enter**.

The rest of this training runs Kubernetes Jobs for you — you `kubectl apply` a manifest and read the logs. Before that, it helps to see the basic unit those Jobs are built from: a **pod**, and get comfortable with the handful of commands you'll use on every pod and Job for the rest of the day.

**In this exercise you will:**

1. Launch a single pod from a YAML manifest.
2. Read its logs.
3. Run a one-off command inside it, then open an interactive shell.
4. Practice the two commands you reach for the moment something looks wrong — `describe` and `get events`.
5. Clean up.

Everything here is deliberately small — one pod, no PVC, no image to build — so the mechanics stay visible. The jet-classifier exercise later in this training reuses every command you learn here, just aimed at a Job instead of a bare pod.

## ⚙️ Set your username

Edit `USER` below, then run the cell.

In [ ]:
export USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/cms-hats/workspace
if [ "$USER" = changeme ]; then echo "⚠️  Edit USER above first, then re-run"; else
  cp yamls/pod-basics.yaml /tmp/pod-basics-${USER}.yaml
  perl -pi -e 's/<username>/$ENV{USER}/g' /tmp/pod-basics-${USER}.yaml
  echo "✅ /tmp/pod-basics-${USER}.yaml ready"
fi


## Launch a pod

`yamls/pod-basics.yaml` is deliberately minimal:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: pod-basics-<username>
  namespace: us-cms
spec:
  containers:
  - name: mypod
    image: ubuntu:22.04
    command: ["sh", "-c", "echo 'Hello from NRP!' && sleep 3600"]
    resources:
      limits:   { memory: 100Mi, cpu: 100m }
      requests: { memory: 100Mi, cpu: 100m }
```

In [ ]:
kubectl apply -n us-cms -f /tmp/pod-basics-${USER}.yaml


In [ ]:
kubectl wait --for=condition=Ready pod/pod-basics-${USER} -n us-cms --timeout=60s


In [ ]:
kubectl get pods -n us-cms


## Read the logs

We sleep briefly first — `Ready` means the container is running, not that it has necessarily flushed its first line of output yet.

In [ ]:
sleep 5
kubectl logs pod-basics-${USER} -n us-cms


<details>
<summary>Expected output</summary>

```text
pod/pod-basics-<username> created

NAME                        READY   STATUS    RESTARTS   AGE
pod-basics-<username>       1/1     Running   0          8s

Hello from NRP!
```
</details>

## Run commands inside the pod

In [ ]:
kubectl exec pod-basics-${USER} -n us-cms -- echo 'Command executed successfully'


**🖥️ Terminal step** — interactive: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-D to exit.

```bash
kubectl exec -it pod-basics-${USER} -n us-cms -- /bin/bash
```

## The debugging trio: describe, events, previous logs

When something doesn't behave the way you expect:

In [ ]:
kubectl describe pod pod-basics-${USER} -n us-cms


In [ ]:
kubectl get events -n us-cms --field-selector involvedObject.name=pod-basics-${USER} --sort-by=.metadata.creationTimestamp


`describe` shows scheduling decisions and container state; `get events` (filtered to just this pod with `--field-selector`) shows its scheduling timeline — pulling the image, mounting volumes, starting the container. Drop the `--field-selector` to see every event in the namespace instead. For a **crashlooping** pod, add `--previous` to read the dead container's logs — `kubectl logs <pod> -n us-cms --previous`. On a healthy pod it just says *"previous terminated container not found"* — that's expected, not an error.

## Clean up

In [ ]:
kubectl delete pod pod-basics-${USER} -n us-cms


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.

In [ ]:
bash check.sh 2
